### Imports and R Environment Setup


In [59]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import rpy2.robjects as ro
from rpy2.robjects import numpy2ri
from rpy2.robjects.conversion import localconverter

# Your specific package imports
import graphical_sampling as gs
from package_sampling.utils import inclusion_probabilities

# Initialize the converter without calling .activate()
# This is the modern, non-deprecated way to handle rpy2 conversion
numpy2ri_converter = numpy2ri.converter
conv = ro.default_converter + numpy2ri_converter

# Initialize R with scoring functions
ro.r("""
    library(sf)
    library(BalancedSampling)
    library(sampling)
    library(WaveSampling)
    library(spbal)
    library(spsurvey)

    calc_r_metrics <- function(coords, probs, sample_mask, sample_idx) {
        # Ensure input types for R functions
        coords_mat <- as.matrix(coords)
        probs_vec <- as.numeric(probs)
        
        # Moran's I via wpik
        W <- wpik(coords_mat, probs_vec)
        diag(W) <- 0
        
        ib_val <- tryCatch(IB(W, sample_mask), error = function(e) Inf)
        sb_val <- tryCatch(sb(probs_vec, coords_mat, sample_idx), error = function(e) Inf)
        sblb_val <- tryCatch(sblb(probs_vec, coords_mat, sample_idx), error = function(e) Inf)
        
        return(c(ib_val, sb_val, sblb_val))
    }
""")

### Optimized Sampling Functions

In [60]:
def run_sampling_design(method, coords, probs, n, num_samples, is_EP):
    N = len(coords)
    
    # 1. Python Methods (Including your Nmcs)
    if method == "Nmcs":
        return gs.sampling.KMeansSpatialSamplingSimple(
            coords, probs, n=n, n_zones=(2, 2), tolerance=2, split_size=0.001
        ).sample(num_samples)

    if method == "Rand":
        samples_idx = np.zeros((num_samples, n), dtype=int)
        for i in range(num_samples):
            print(np.random.choice(N, n, replace=False))
            samples_idx[i] = np.random.choice(N, n, replace=False)
        return samples_idx

    # 2. R Methods
    samples_idx = np.zeros((num_samples, n), dtype=int)
    with localconverter(conv):
        ro.globalenv['coords'] = coords
        ro.globalenv['probs'] = probs
        ro.globalenv['n'] = n
        
        if method in ["HIP", "GRTS"]:
            ro.r("pts <- st_as_sf(data.frame(x=coords[,1], y=coords[,2]), coords=c('x','y'))")
            if not is_EP: ro.r("pts$probs <- probs")

        for i in range(num_samples):
            if method == "Lopi":
                samples_idx[i] = np.array(ro.r("BalancedSampling::lpm2(probs, coords)")) - 1
                print(sample_idx)
            elif method == "Wave":
                mask = ro.r("WaveSampling::wave(coords, probs)")
                samples_idx[i] = np.where(np.array(mask).astype(bool))[0]
            elif method == "Maxe":
                mask = ro.r("sampling::UPmaxentropy(probs)")
                samples_idx[i] = np.where(np.array(mask).astype(bool))[0]
            elif method == "HIP":
                ro.r(f"res <- spbal::HIP(population = pts, n = {n})")
                samples_idx[i] = np.array(ro.r("as.integer(rownames(res$sample))")) - 1
            elif method == "GRTS":
                aux = ', aux_var = "probs"' if not is_EP else ""
                ro.r(f"res <- spsurvey::grts(sframe = pts, n_base = {n} {aux})")
                samples_idx[i] = np.array(ro.r("as.numeric(rownames(res$sites_base))")) - 1
                
    return samples_idx

def calculate_ht_estimator(y, sample_indices, probs):
    sample_y = y[sample_indices]
    sample_probs = probs[sample_indices]
    return np.sum(sample_y / sample_probs)

### Metrics and Spread Calculation

In [61]:
def calculate_all_scores(coords, probs, sample_idx, n, N, density_measure, y_val):
    # 1. Density Score (Python - graphical_sampling)
    dens_score, _ = density_measure.score(sample_idx.reshape(1, -1))
    
    # 2. HT Estimator Total
    ht_val = np.sum(y_val[sample_idx] / probs[sample_idx])
    
    # 3. R Metrics (Spatial Balance)
    sample_mask = np.zeros(N, dtype=int)
    sample_mask[sample_idx] = 1
    
    with localconverter(conv):
        ro.globalenv['sample_mask'] = sample_mask
        ro.globalenv['s_idx_r'] = sample_idx + 1
        r_results = ro.r("calc_r_metrics(coords, probs, sample_mask, s_idx_r)")
        
    # Order: Density, Voronoi (sb), Moran (ib), Local Balance (sblb), HT_Total
    print('injooo:dens_score', dens_score)
    return dens_score[0], r_results[1], r_results[0], r_results[2], ht_val

### The Main Execution Loop

In [63]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

# --- Configuration ---
folder = "/config/ws/graphical-sampling/populations"
results_folder = "data_samples"
pop_names = ["random_uneq"]
sample_cnt = 10 
n_size = 5

os.makedirs(results_folder, exist_ok=True)

class PopulationObj:
    def __init__(self, coords, probs):
        self.coords, self.probs = coords, probs
        self.N, self.indices = len(coords), np.arange(len(coords))

for name in pop_names:
    # 1. Load and Prep Data
    file_path = os.path.join(folder, f"{name}_N=100.csv")
    
    df = pd.read_csv(file_path)
    coords = df[["x", "y"]].values.astype(float)
    
    # Calculate inclusion probabilities immediately
    pik = inclusion_probabilities(df["prob"].values, n_size)
    n = int(np.round(np.sum(pik)))
    is_EP = np.allclose(pik, pik[0])

    print(f"\n--- Processing {name} (N={len(pik)}, n={n}) ---")

    # 2. Setup Density Measure
    # FIXED: Use np.ptp() and np.min() instead of array methods for NumPy 2.0+
    scaled_coords = (coords - np.min(coords, axis=0)) / (np.ptp(coords, axis=0) + 1e-9)
    pop_wrapped = PopulationObj(scaled_coords, pik)
    
    density_measure = gs.measure.Density(pop_wrapped, n, 0.001)
    

    # 3. Sampling and Scoring
    all_data = []
    methods = ["Rand", "Lopi"] 

    for m in methods:
        print(f"Running {m}...")
        samples = run_sampling_design(m, coords, pik, n, sample_cnt, is_EP)
        
        for i in tqdm(range(sample_cnt), desc=f"Scoring {m}"):
            s_idx = samples[i]
            print('injaaaaaaaaa')
            metrics = calculate_all_scores(coords, pik, s_idx, n, len(pik), density_measure, 
                                           df["y"].values)
            
            all_data.append([m] + list(metrics))

    # 4. Results Processing
    if not all_data: continue

    res_df = pd.DataFrame(all_data, columns=["Method", "Density", "Voronoi", "Moran", "Local_Balance", "HT_Total"])
    summary = res_df.groupby("Method")["HT_Total"].agg(['var', 'mean'])
    
    if "Rand" in summary.index:
        summary["Efficiency"] = summary.loc["Rand", "var"] / summary["var"].replace(0, np.nan)

    print(summary)
    res_df.to_csv(os.path.join(results_folder, f"final_results_{name}.csv"), index=False)


--- Processing random_uneq (N=100, n=5) ---
Running Rand...
[25 41 37 32 18]
[44 15 45 86 36]
[39 57 67 87 50]
[44  3  1 68 47]
[78 98 27 45 18]
[96 32 94 21 98]
[ 9 69 45  8 80]
[17 77 85 81 51]
[84 63 90 47 17]
[59 65 45 46 57]


Scoring Rand:   0%|          | 0/10 [00:00<?, ?it/s]

injaaaaaaaaa


TypeError: unsupported operand type(s) for -: 'float' and 'NoneType'

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

# --- Configuration ---
folder = "/config/ws/graphical-sampling/populations"
pop_names = ["random_uneq"]
sample_cnt = 10  # Set to 1000 for final production run
n_size = 5
results_folder = "data_samples"
os.makedirs(results_folder, exist_ok=True)


# -------------------------------------------------
# Helper class for Density measure requirements
# -------------------------------------------------
class PopulationObj:
    def __init__(self, coords, probs):
        self.coords = coords
        self.probs = probs
        self.N = len(coords)
        self.indices = np.arange(self.N)


# -------------------------------------------------
# Main Loop
# -------------------------------------------------
for name in pop_names:

    # 1. Load Data
    file_path = os.path.join(folder, f"{name}_N=100.csv")
    if not os.path.exists(file_path):
        print(f"Skipping {name}: File not found at {file_path}")
        continue

    df = pd.read_csv(file_path)
    print(df)
    coords = df[["x", "y"]].values.astype(float)
    pik = inclusion_probabilities(probs, n_size)
    probs = pik

    # 2. Setup Variables
    y_val = (
        df["y"].values
        if "y" in df.columns
        else (coords[:, 0] ** 2 + coords[:, 1] ** 2)
    )

    N = len(probs)

    # Compute sample size
    n = int(np.sum(pik))
    print('n = ', n)
    is_EP = np.allclose(probs, probs[0])

    print(f"\nProcessing {name}: N={N}, n={n} (sum={n_sum:.2f})")

    # -------------------------------------------------
    # 3. Initialize Density Measure
    # -------------------------------------------------

    # Safe min-max scaling
    coord_min = np.min(coords, axis=0)
    coord_range = np.ptp(coords, axis=0)
    coord_range[coord_range == 0] = 1.0  # avoid division by zero

    scaled_coords = (coords - coord_min) / coord_range

    pop_wrapped = PopulationObj(scaled_coords, probs)

    try:
        density_measure = gs.measure.Density(pop_wrapped, n, 0.1)
        print(density_measure)
    except Exception as e:
        print(f"Density Measure failed to initialize for {name}: {e}")
        continue

    # -------------------------------------------------
    # 4. Iterate through Sampling Designs
    # -------------------------------------------------

    all_data = []
    # methods = ["Rand", "Nmcs", "Lopi", "Wave", "GRTS"]

    methods = ["Rand", "Lopi",]
    
    for m in methods:
        print(f"\n--- Running Method: {m} ---")

        try:
            # Generate samples
            samples = run_sampling_design(
                m, coords, probs, n, sample_cnt, is_EP
            )

            samples = np.asarray(samples)

            if samples.shape[0] != sample_cnt:
                print(f"Warning: Method {m} returned unexpected sample count")
                continue

            # Score each sample
            for i in tqdm(range(sample_cnt), desc=f"Scoring {m}"):

                sample_idx = samples[i]

                # Safety check
                if len(sample_idx) != n:
                    continue

                scores = calculate_all_scores(
                    coords,
                    probs,
                    sample_idx,
                    n,
                    N,
                    density_measure,
                    y_val,
                )

                all_data.append([m] + list(scores))

        except Exception as e:
            print(f"Error running method {m}: {e}")

    # -------------------------------------------------
    # 5. Process Results & Efficiency
    # -------------------------------------------------

    if not all_data:
        print("No data collected. Skipping summary.")
        continue

    res_df = pd.DataFrame(
        all_data,
        columns=[
            "Method",
            "Density",
            "Voronoi",
            "Moran",
            "Local_Balance",
            "HT_Total",
        ],
    )

    summary_stats = res_df.groupby("Method").agg({
        "Density": ["mean", "std"],
        "Voronoi": ["mean", "std"],
        "Moran": ["mean", "std"],
        "Local_Balance": ["mean", "std"],
        "HT_Total": ["var", "mean"],
    })

    # Efficiency relative to Random
    if "Rand" in summary_stats.index:
        srs_var = summary_stats.loc["Rand", ("HT_Total", "var")]
        method_vars = summary_stats[("HT_Total", "var")]

        # Avoid division by zero
        method_vars = method_vars.replace(0, np.nan)

        summary_stats["Efficiency"] = srs_var / method_vars

    # -------------------------------------------------
    # 6. Save and Display
    # -------------------------------------------------

    print(f"\n--- Results for {name} ---")
    print(summary_stats)

    output_path = os.path.join(
        results_folder, f"final_results_{name}.csv"
    )

    res_df.to_csv(output_path, index=False)

    print(f"Saved results to {output_path}")


           x         y      prob
0   0.378327  0.958959  0.007693
1   0.676830  0.333250  0.013763
2   0.041106  0.283703  0.001152
3   0.360157  0.643663  0.007195
4   0.659020  0.826632  0.013249
..       ...       ...       ...
95  0.333295  0.317297  0.006813
96  0.769206  0.200533  0.014903
97  0.452098  0.456165  0.009776
98  0.067824  0.343011  0.001653
99  0.144398  0.901450  0.002818

[100 rows x 3 columns]
n =  5

Processing random_uneq: N=100, n=5 (sum=1.05)

--- Running Method: Rand ---
[42 85 19 16 96]
[76 58 96 27 48]
[53 16 21 84 44]
[68  3 99 51 79]
[93 95 84 82  1]
[16  9 90 24 32]
[41 99 46 95 49]
[83 98 37 24 99]
[22 35 24 25  3]
[ 3 33 51 91 90]


Scoring Rand:   0%|          | 0/10 [00:00<?, ?it/s]


Error running method Rand: unsupported operand type(s) for -: 'float' and 'NoneType'

--- Running Method: Lopi ---
[51 29 85 80 15]
[51 29 85 80 15]
[51 29 85 80 15]
[51 29 85 80 15]
[51 29 85 80 15]
[51 29 85 80 15]
[51 29 85 80 15]
[51 29 85 80 15]
[51 29 85 80 15]
[51 29 85 80 15]


Scoring Lopi:   0%|          | 0/10 [00:00<?, ?it/s]

Error running method Lopi: unsupported operand type(s) for -: 'float' and 'NoneType'
No data collected. Skipping summary.
